In [1]:
import numpy as np
from scipy.stats import norm
from typing import Union, List

def black_scholes_eur_put(
    r: float, # risk-free interest rate
    T: float, # time to maturity in years
    S0: float, # current stock price
    sigma: float, # stock volatility
    K: Union[float, List[float], np.ndarray] # strike price or an array of strike prices
) -> np.ndarray:

    # Check input conditions.
    assert sigma > 0, "Volatility must be positive."
    assert T > 0, "Time to maturity must be positive."
    assert S0 > 0, "Stock price must be positive."

    # Convert strike prices into a NumPy array.
    K_vec = np.atleast_1d(np.asarray(K, dtype=float))

    assert np.all(K_vec > 0), "Strike prices must be positive."

    # Calculate d1 and d2.
    d1_vec = (
        np.log(S0 / K_vec)
        + (r + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2_vec = d1_vec - sigma * np.sqrt(T)

    # Calculate cumulative probabilities for -d1 and -d2.
    N_minus_d1_vec = norm.cdf(-d1_vec)
    N_minus_d2_vec = norm.cdf(-d2_vec)

    # Calculate the European put option price.
    put_price = (
        K_vec * np.exp(-r * T) * N_minus_d2_vec
        - S0 * N_minus_d1_vec
    )

    # Return the put option price.
    return put_price
    
def black_scholes_eur_call(
    r: float, # risk-free interest rate
    T: float,  #time to maturity in years
    S0: float, #current stock price
    sigma: float, #stock volatility
    K: Union[float, List[float], np.ndarray] #strike price or an array of strike prices
) -> dict:

    # Check input conditions.
    assert sigma > 0, "Volatility must be positive."
    assert T > 0, "Time to maturity must be positive."
    assert S0 > 0, "Stock price must be positive."

    #Convert strike prices into a NumPy array.
    K_vec = np.atleast_1d(np.asarray(K, dtype=float))

    # Check that all strike prices are positive.
    assert np.all(K_vec > 0), "Strike prices must be positive."

    # Calculate d1 and d2
    d1_vec = (
        np.log(S0 / K_vec)
        + (r + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2_vec = d1_vec - sigma * np.sqrt(T)

    # Calculate standard normal cumulative probabilities.
    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # Calculate the European call option price.
   
    call_price = (
        S0 * N_d1_vec
        - K_vec * np.exp(-r * T) * N_d2_vec
    )

    # Calculate the standard normal density at d1.
    pdf_d1_vec = norm.pdf(d1_vec)

    # Calculate Delta: sensitivity to the stock price.
    delta = N_d1_vec

    # Calculate Gamma
    gamma = (
        pdf_d1_vec / (S0 * sigma * np.sqrt(T))
    )

    # Calculate Vega: sensitivity to volatility.

    vega = S0 * pdf_d1_vec * np.sqrt(T)

    # Calculate Theta: sensitivity to calendar time.
    
    theta = (
        -(S0 * sigma * pdf_d1_vec) / (2 * np.sqrt(T))
        - r * K_vec * np.exp(-r * T) * N_d2_vec
    )

    #Calculate Rho: sensitivity to the risk-free rate.
    rho = K_vec * T * np.exp(-r * T) * N_d2_vec

    # Return the price and all Greeks in a dictionary.
    return {
        'price': call_price,
        'delta': delta,
        'gamma': gamma,
        'vega': vega,
        'theta': theta,
        'rho': rho
    }


In [2]:

r = 0.05
T = 1.0
S0 = 100.0
sigma = 0.20
K_values = [80, 90, 100, 110, 120]

# Calculate call and put prices.
call_prices = black_scholes_eur_call(r, T, S0, sigma, K_values)['price']
put_prices = black_scholes_eur_put(r, T, S0, sigma, K_values)

# Calculate both sides of the parity equation.
left_side = call_prices - put_prices
right_side = S0 - np.array(K_values) * np.exp(-r * T)

# Compare the results.
print(f"{'K':>5} {'Left Side':>15} {'Right Side':>15} {'Difference':>15}")

for i, K in enumerate(K_values):
    print(f"{K:5.0f} {left_side[i]:15.6f} {right_side[i]:15.6f} {(left_side[i] - right_side[i]):15.10f}")

print("Put-Call Parity holds:", np.allclose(left_side, right_side))


    K       Left Side      Right Side      Difference
   80       23.901646       23.901646    0.0000000000
   90       14.389352       14.389352    0.0000000000
  100        4.877058        4.877058    0.0000000000
  110       -4.635237       -4.635237   -0.0000000000
  120      -14.147531      -14.147531    0.0000000000
Put-Call Parity holds: True
